In [53]:
import random
from collections import defaultdict

In [ ]:
#Universal Constants

STARTING_REWARD = 10.5  
    # Decreasing this reduces training time but decreases exploration
    # Could make the exploration function into a softmin later
    
TRAINING_N = 10000
    # Number of training iterations

DEFAULT_PRICE = 10
    #This is only useful before we have read in prices

In [61]:
class Actor:
    #Create a new actor with predefined memory
    def __init__(self, memory = None):
        if memory:
            self.memory = memory
            self.energy = "Empty"
        else:
            
            def default_actions():
                return {"Charge": STARTING_REWARD, "Discharge": STARTING_REWARD, "Hold": STARTING_REWARD}

            # Memory that creates missing states on the fly
            memory = defaultdict(default_actions)
            self.memory = memory
            self.energy = "Empty"
            print(memory)

    def round_state(self, state):
        bucketized_state = []
        for price in state:
            bucketized_state.append(round(price,-1))
        return tuple(bucketized_state)


    #Choose an action given the state space x1, x2
    def choose_action(self, state):
        #bucketization
        state = self.round_state(state)
        #Remember what has happened in past situations in this state space
        data = self.memory[state]

        if self.energy == "Empty":
            #Only possible actions here are charge or hold
            charge_val = data["Charge"]
            hold_val = data["Hold"]
            total = charge_val + hold_val
            confidence = charge_val / total
            if random.uniform(0,1) > confidence:
                action = "Charge"
                self.energy = "Full"
            else:
                action = "Hold"

        elif self.energy == "Full":
            #The only possible actions here are discharge or hold
            discharge_val = data["Discharge"]
            hold_val = data["Hold"]
            total = discharge_val + hold_val
            confidence = discharge_val / total
            if random.uniform(0,1) > confidence:
                action = "Discharge"
                self.energy = "Empty"
            else:
                action = "Hold"
        else:
            raise Exception("Energy level was neither full nor empty")
        
        return action, confidence

    def update_reward(self,state,action,reward):
        state = self.round_state(state)
        self.memory[state][action] += reward

### Ideas:
- schedule out the buy and sell
    - action space is ["Charge", "Hold", "Hold", "Discharge", ...]

In [72]:
#train
def run(type, memory=None):
    if type == "Train":
        n = TRAINING_N
        actor = Actor()
    elif type == "Continue Train":
        n = TRAINING_N
        actor = Actor(memory)
    elif type == "Test":
        n = 1
        actor = Actor(memory)
    
    for i in range(n):
        #Building the state as an hourly schedule 1-24
        state = []
        for h in range(24):
            state.append(random.uniform(-10,100))
        state = tuple(state)
        action, confidence = actor.choose_action(state)
        
        #Buying electricity
        if action == "Charge":
            reward = -1 * state[0]
        #Not buying or selling electricity
        elif action == "Hold":
            reward = 0
        #Selling Electricity
        else:
            reward = state[0]

        actor.update_reward(state,action,reward)
        print(action)
        
    if type == "Test":
        print(state, action, confidence, reward)
        print(actor.memory)
    return actor.memory




In [73]:
memory = run(type = "Train")

defaultdict(<function Actor.__init__.<locals>.default_actions at 0x00000256ADE4FAC0>, {})
Charge
Hold
Discharge
Charge
Hold
Hold
Hold
Discharge
Hold
Hold
Charge
Discharge
Hold
Hold
Hold
Hold
Hold
Charge
Hold
Hold
Hold
Discharge
Charge
Discharge
Charge
Hold
Discharge
Hold
Hold
Charge
Discharge
Hold
Charge
Discharge
Charge
Discharge
Hold
Charge
Discharge
Hold
Hold
Hold
Charge
Hold
Hold
Hold
Hold
Discharge
Hold
Hold
Charge
Hold
Discharge
Charge
Hold
Discharge
Hold
Charge
Hold
Discharge
Hold
Hold
Hold
Hold
Charge
Discharge
Hold
Hold
Hold
Charge
Hold
Hold
Discharge
Hold
Hold
Charge
Hold
Discharge
Hold
Hold
Charge
Hold
Discharge
Charge
Discharge
Charge
Discharge
Charge
Discharge
Charge
Hold
Hold
Discharge
Charge
Hold
Discharge
Charge
Hold
Discharge
Charge
Discharge
Charge
Hold
Discharge
Hold
Hold
Charge
Hold
Hold
Discharge
Charge
Discharge
Hold
Hold
Hold
Charge
Discharge
Hold
Charge
Discharge
Hold
Charge
Hold
Hold
Hold
Hold
Hold
Discharge
Charge
Discharge
Charge
Hold
Discharge
Charge
Dischar

In [70]:
memory

defaultdict(<function __main__.Actor.__init__.<locals>.default_actions()>,
            {(60.0,
              40.0,
              10.0,
              80.0,
              10.0,
              40.0,
              90.0,
              90.0,
              -0.0,
              80.0,
              20.0,
              100.0,
              -0.0,
              70.0,
              20.0,
              10.0,
              90.0,
              10.0,
              90.0,
              0.0,
              70.0,
              -0.0,
              70.0,
              60.0): {'Charge': -48.165130716436636,
              'Discharge': 10.5,
              'Hold': 10.5},
             (70.0,
              50.0,
              30.0,
              60.0,
              10.0,
              70.0,
              70.0,
              30.0,
              70.0,
              60.0,
              -10.0,
              60.0,
              -10.0,
              90.0,
              70.0,
              20.0,
              70.0,
        

In [71]:
memory = run(type = "Test", memory=memory)

(59.71502285288828, 13.59152164375628, -1.7560468931622388, 15.677767653263121, 59.731163299840134, 30.279836014760953, 1.4841609893388181, 43.4947478221692, 8.122468613192815, -9.929749928674058, 80.6573612436281, 79.64891090448882, 64.85264308410021, 97.7929191690542, 95.36233201741067, 15.328698641209407, 82.58254206973244, 83.32887048197291, 5.506784445661399, 26.73584244471118, 19.763822860021847, 0.13771585366750116, 36.214202602715304, 88.11141161148107) Charge 0.5 -59.71502285288828
defaultdict(<function Actor.__init__.<locals>.default_actions at 0x00000256AF00BD90>, {(60.0, 40.0, 10.0, 80.0, 10.0, 40.0, 90.0, 90.0, -0.0, 80.0, 20.0, 100.0, -0.0, 70.0, 20.0, 10.0, 90.0, 10.0, 90.0, 0.0, 70.0, -0.0, 70.0, 60.0): {'Charge': -48.165130716436636, 'Discharge': 10.5, 'Hold': 10.5}, (70.0, 50.0, 30.0, 60.0, 10.0, 70.0, 70.0, 30.0, 70.0, 60.0, -10.0, 60.0, -10.0, 90.0, 70.0, 20.0, 70.0, 30.0, 30.0, -0.0, 50.0, 80.0, 100.0, 70.0): {'Charge': 10.5, 'Discharge': 77.53944075992948, 'Hold':